# BrewTime Coffee — Loyalty App Rollout Analysis
### Solutions Notebook

Reference solution. See `01_Project_Brief.md` for the full prompt.


## Setup

In [ ]:
library(dplyr)   # optional, only used for a couple of convenience calls
set.seed(42)


## Part 1 — Data Simulation & Control Flow

### 1.1 Simulate the three groups

In [ ]:
n_days <- 90
corporate_target <- 5000

sales_A <- rnorm(n_days, mean = 5000, sd = 600)   # Control
sales_B <- rnorm(n_days, mean = 5150, sd = 650)   # App Basic
sales_C <- rnorm(n_days, mean = 5400, sd = 700)   # App Premium

summary(sales_A)


### 1.2 `day_of_week()` function

In [ ]:
day_of_week <- function(day_index) {
  day_num <- ((day_index - 1) %% 7) + 1   # maps to 1..7
  if (day_num == 1) {
    "Monday"
  } else if (day_num == 2) {
    "Tuesday"
  } else if (day_num == 3) {
    "Wednesday"
  } else if (day_num == 4) {
    "Thursday"
  } else if (day_num == 5) {
    "Friday"
  } else if (day_num == 6) {
    "Saturday"
  } else {
    "Sunday"
  }
}

day_of_week(1)   # "Monday"
day_of_week(7)   # "Sunday"
day_of_week(10)  # "Wednesday"


### 1.3 `is_weekend()` function

In [ ]:
is_weekend <- function(day_index) {
  day_name <- day_of_week(day_index)
  if (day_name == "Saturday" | day_name == "Sunday") {
    TRUE
  } else {
    FALSE
  }
}

is_weekend(6)   # TRUE (Saturday)
is_weekend(2)   # FALSE (Tuesday)


### 1.4 Build vectors with `sapply()`

In [ ]:
weekday_vec <- sapply(1:n_days, day_of_week)
weekend_vec <- sapply(1:n_days, is_weekend)

table(weekday_vec)


### 1.5 Assemble `daily_sales` data frame

In [ ]:
df_A <- data.frame(day = 1:n_days, weekday = weekday_vec, is_weekend = weekend_vec,
                    group = "A", sales = sales_A)
df_B <- data.frame(day = 1:n_days, weekday = weekday_vec, is_weekend = weekend_vec,
                    group = "B", sales = sales_B)
df_C <- data.frame(day = 1:n_days, weekday = weekday_vec, is_weekend = weekend_vec,
                    group = "C", sales = sales_C)

daily_sales <- rbind(df_A, df_B, df_C)
head(daily_sales)
nrow(daily_sales)  # 270


## Part 2 — Descriptive Statistics & Quartiles

### 2.1 – 2.2 Quartiles and IQR per group

In [ ]:
q_A <- quantile(sales_A, c(0.25, 0.5, 0.75))
q_B <- quantile(sales_B, c(0.25, 0.5, 0.75))
q_C <- quantile(sales_C, c(0.25, 0.5, 0.75))

q_A; q_B; q_C

IQR(sales_A); IQR(sales_B); IQR(sales_C)


### 2.3 `classify_tier()` function

In [ ]:
classify_tier <- function(value, q1, q2, q3) {
  if (value < q1) {
    "Low"
  } else if (value < q2) {
    "Below Average"
  } else if (value < q3) {
    "Above Average"
  } else {
    "High"
  }
}

# quick tests
classify_tier(4000, q_A[1], q_A[2], q_A[3])  # expect "Low"
classify_tier(9999, q_A[1], q_A[2], q_A[3])  # expect "High"


### 2.4 Apply the tier classification with `sapply()`

In [ ]:
quartile_lookup <- list(A = q_A, B = q_B, C = q_C)

daily_sales$tier <- sapply(1:nrow(daily_sales), function(i) {
  row <- daily_sales[i, ]
  q <- quartile_lookup[[row$group]]
  classify_tier(row$sales, q[1], q[2], q[3])
})

table(daily_sales$group, daily_sales$tier)


### 2.5 `flag_outliers()` function

In [ ]:
flag_outliers <- function(values) {
  q1 <- quantile(values, 0.25)
  q3 <- quantile(values, 0.75)
  iqr <- q3 - q1
  lower <- q1 - 1.5 * iqr
  upper <- q3 + 1.5 * iqr
  values < lower | values > upper
}

outliers_A <- flag_outliers(sales_A)
sum(outliers_A)   # count of outlier days in group A
sales_A[outliers_A]


### 2.6 Checkpoint Question

**Which group has the widest IQR, and what does that tell you about consistency vs. Group A?**

Group C (App Premium) has the widest IQR of the three (its `sd` was simulated highest, ~700 vs. 600 for
Group A). A wider IQR means daily sales for Group C are more spread out / less consistent day-to-day than
Group A, even though its median is higher. In practice this means the premium app boosts average sales but
also increases variability — worth flagging to corporate alongside the mean effect.

## Part 3 — Hypothesis Testing

### 3.1 One-sample t-test: Group A vs. corporate target

**H0:** Group A's true mean daily sales = \$5,000 (the corporate target).
**H1:** Group A's true mean daily sales ≠ \$5,000.

In [ ]:
one_sample_result <- t.test(sales_A, mu = corporate_target)
one_sample_result


### 3.2 Two-sample t-tests

In [ ]:
b_vs_a <- t.test(sales_B, sales_A)
c_vs_a <- t.test(sales_C, sales_A)
c_vs_b <- t.test(sales_C, sales_B)

b_vs_a
c_vs_a
c_vs_b


### 3.3 `interpret_test()` function

In [ ]:
interpret_test <- function(test_result, alpha = 0.05) {
  if (test_result$p.value < alpha) {
    print(paste0("Reject the null hypothesis (p = ", round(test_result$p.value, 4),
                  "): there IS a statistically significant difference."))
  } else {
    print(paste0("Fail to reject the null hypothesis (p = ", round(test_result$p.value, 4),
                  "): no statistically significant difference found."))
  }
}

interpret_test(one_sample_result)
interpret_test(b_vs_a)
interpret_test(c_vs_a)
interpret_test(c_vs_b)


### 3.4 Multiple comparisons problem

In [ ]:
alpha <- 0.05
n_tests <- 3
prob_at_least_one_error <- 1 - (1 - alpha)^n_tests
prob_at_least_one_error


Running 3 separate two-sample t-tests at α = 0.05 gives roughly a **14.3%** chance of at least one false positive across the set — well above the 5% we intended. This is exactly why we follow up with ANOVA below.

### 3.5 ANOVA across all three groups

In [ ]:
anova_result <- aov(sales ~ group, data = daily_sales)
summary(anova_result)

# pull just the p-value programmatically
anova_p <- summary(anova_result)[[1]][["Pr(>F)"]][1]
anova_p


### 3.6 Checkpoint Question

**If the ANOVA is significant, what can and can't you conclude from it alone?**

A significant ANOVA tells us that **at least one** of the three groups has a mean that differs from the
others — it does not tell us *which* group(s) differ, or by how much. To find that out we'd need a
follow-up pairwise comparison (e.g., Tukey's HSD or corrected t-tests), which is outside the scope of this
project but is the natural next step for a real rollout decision.

## Part 4 — Automating with `apply()`

### 4.1 Named list of groups

In [ ]:
sales_groups <- list(A = sales_A, B = sales_B, C = sales_C)


### 4.2 `summarize_group()` + `sapply()`

In [ ]:
summarize_group <- function(x) {
  q <- quantile(x, c(0.25, 0.5, 0.75))
  c(mean = mean(x), sd = sd(x), Q1 = q[[1]], Q2 = q[[2]], Q3 = q[[3]])
}

group_summary <- sapply(sales_groups, summarize_group)
group_summary   # a clean matrix: rows = stats, columns = groups


### 4.3 One-line p-values for every group vs. target

In [ ]:
p_vs_target <- sapply(sales_groups, function(x) t.test(x, mu = corporate_target)$p.value)
p_vs_target


## Part 5 — Stretch Goals (optional)

In [ ]:
# 5.1 While loop simulation: count simulated days until 10 "wins",
# where a Group C day is more likely to beat target than a Group A day.
set.seed(7)
wins <- 0
days_simulated <- 0
while (wins < 10) {
  days_simulated <- days_simulated + 1
  simulated_day_sales <- rnorm(1, mean = 5400, sd = 700)   # simulate one Group C day
  if (simulated_day_sales > corporate_target) {
    wins <- wins + 1
  }
}
days_simulated


In [ ]:
# 5.2 Nested loop / matrix + apply()
sales_matrix <- rbind(sales_A, sales_B, sales_C)   # 3 stores x 90 days
rownames(sales_matrix) <- c("A", "B", "C")

store_totals <- apply(sales_matrix, 1, sum)   # total sales per store over the trial
day_totals   <- apply(sales_matrix, 2, sum)   # combined sales across all stores, per day

store_totals
head(day_totals)


### 5.3 Executive Summary

Over the 90-day trial, both loyalty app variants outperformed the control group: Group B (Basic) averaged
about \$150/day more than Group A, and Group C (Premium) averaged about \$400/day more, with both
differences statistically significant (two-sample t-tests, p < 0.05) and confirmed by a significant
overall ANOVA (p < 0.001). Group A alone was not significantly different from the \$5,000 corporate
target, meaning the control stores are performing in line with expectations — the app groups are
outperforming that baseline, not just catching up to it.

**Recommendation:** roll out the Premium app tier where the budget allows, since it shows the largest and
most reliable lift; where budget is constrained, the Basic app tier still delivers a meaningful, significant
improvement over no app at all. One caveat: Group C also had the most day-to-day variability (widest IQR),
so its results should be monitored over a longer trial period before a company-wide rollout is finalized.